# Lab 02 - Homework


In [53]:
import os
import gzip
import shutil
from urllib.request import urlretrieve

In [ ]:
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import plotly.express as px

In [55]:
os.makedirs('data', exist_ok=True)

### 1. Download data

In [56]:
# Source files and their URLs
sources = {
    "tariff-distances-2022-01.csv" : "https://opendata.rijdendetreinen.nl/public/tariff-distances/tariff-distances-2022-01.csv",
    "stations-2023-09.csv" : "https://opendata.rijdendetreinen.nl/public/stations/stations-2023-09.csv",
    "disruptions-2011.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2011.csv",
    "disruptions-2012.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2012.csv",
    "disruptions-2013.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2013.csv",
    "disruptions-2014.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2014.csv",
    "disruptions-2015.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2015.csv",
    "disruptions-2016.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2016.csv",
    "disruptions-2017.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2017.csv",
    "disruptions-2018.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2018.csv",
    "disruptions-2019.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2019.csv",
    "disruptions-2020.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2020.csv",
    "disruptions-2021.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2021.csv",
    "disruptions-2022.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2022.csv",
    "disruptions-2023.csv" : "https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-2023.csv",
    "services-2019.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2019.csv.gz",
    "services-2020.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2020.csv.gz",
    "services-2021.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2021.csv.gz",
    "services-2022.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2022.csv.gz",
    "services-2023.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2023.csv.gz",
    "services-2024.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2024.csv.gz",
    "services-2025-09.csv.gz" : "https://opendata.rijdendetreinen.nl/public/services/services-2025-09.csv.gz"
}

# Target files after ungzip, merge and rename
files = {
    "tariff-distances-2022.csv" : "data/tariff-distances-2022.csv",
    "stations-2023.csv" : "data/stations-2023.csv",
    "disruptions-2011-2023.csv" : "data/disruptions-2011-2023.csv",
    "services-2019-2025.csv" : "data/services-2019-2025.csv"
}

In [57]:
# Download all source files into data/source_files
downloaded = []

for name, url in sources.items():
    target = os.path.join('data', 'source_files', name)
    tmp_target = target + '.part'

    if os.path.exists(target):
        print(f'Skipping download, file exists: {target}')
        downloaded.append(target)
        continue

    print(f'Downloading {url} to {target}')
    urlretrieve(url, tmp_target)

    shutil.move(tmp_target, target)
    downloaded.append(target)

Skipping download, file exists: data\source_files\tariff-distances-2022-01.csv
Skipping download, file exists: data\source_files\stations-2023-09.csv
Skipping download, file exists: data\source_files\disruptions-2011.csv
Skipping download, file exists: data\source_files\disruptions-2012.csv
Skipping download, file exists: data\source_files\disruptions-2013.csv
Skipping download, file exists: data\source_files\disruptions-2014.csv
Skipping download, file exists: data\source_files\disruptions-2015.csv
Skipping download, file exists: data\source_files\disruptions-2016.csv
Skipping download, file exists: data\source_files\disruptions-2017.csv
Skipping download, file exists: data\source_files\disruptions-2018.csv
Skipping download, file exists: data\source_files\disruptions-2019.csv
Skipping download, file exists: data\source_files\disruptions-2020.csv
Skipping download, file exists: data\source_files\disruptions-2021.csv
Skipping download, file exists: data\source_files\disruptions-2022.cs

In [58]:
# Ungzip .gz files from data/source_files into data/
# and copy non-gz files from data/source_files into data/
src_dir = os.path.join('data', 'source_files')

for path in list(downloaded):
    filename = os.path.basename(path)

    if filename.endswith('.gz'):
        out_name = filename[:-3]
        out_path = os.path.join('data', out_name)

        print(f'Unpacking {path} to {out_path}')
        with gzip.open(path, 'rb') as f_in, open(out_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    else:
        # Non-gz files: copy from source_files to data/ 
        out_path = os.path.join('data', filename)
        print(f'Copying {path} to {out_path}')
        shutil.copy2(path, out_path)

Copying data\source_files\tariff-distances-2022-01.csv to data\tariff-distances-2022-01.csv
Copying data\source_files\stations-2023-09.csv to data\stations-2023-09.csv
Copying data\source_files\disruptions-2011.csv to data\disruptions-2011.csv
Copying data\source_files\disruptions-2012.csv to data\disruptions-2012.csv
Copying data\source_files\disruptions-2013.csv to data\disruptions-2013.csv
Copying data\source_files\disruptions-2014.csv to data\disruptions-2014.csv
Copying data\source_files\disruptions-2015.csv to data\disruptions-2015.csv
Copying data\source_files\disruptions-2016.csv to data\disruptions-2016.csv
Copying data\source_files\disruptions-2017.csv to data\disruptions-2017.csv
Copying data\source_files\disruptions-2018.csv to data\disruptions-2018.csv
Copying data\source_files\disruptions-2019.csv to data\disruptions-2019.csv
Copying data\source_files\disruptions-2020.csv to data\disruptions-2020.csv
Copying data\source_files\disruptions-2021.csv to data\disruptions-2021.

In [59]:
# Helper function to concatenate CSV files
def concat_csv(sources_list, dest_path):

    first = True
    with open(dest_path, 'wb') as f_out:
        for src in sources_list:
            print(f'Appending {src} to {dest_path}')
            with open(src, 'rb') as f_in:
                if first:
                    # Write whole file (including header)
                    shutil.copyfileobj(f_in, f_out)
                    first = False
                else:
                    # Skip header line (read it so we can discard it)
                    header = f_in.readline()
                    shutil.copyfileobj(f_in, f_out)

In [60]:
# List disruption and service files
disruption_files = []
service_files = []

for path in os.listdir('data'):
    full_path = os.path.join('data', path)

    if path.startswith('disruptions-'):
        disruption_files.append(full_path)

    if path.startswith('services-'):
        service_files.append(full_path)

# Concat files
concat_csv(disruption_files, files['disruptions-2011-2023.csv'])
concat_csv(service_files, files['services-2019-2025.csv'])

# Move single files (tariff and stations) to their targets
shutil.move(os.path.join('data','tariff-distances-2022-01.csv'), 
            files['tariff-distances-2022.csv'])
shutil.move(os.path.join('data','stations-2023-09.csv'), 
            files['stations-2023.csv'])

# Remove intermediary files
for f in disruption_files + service_files:
    os.remove(f)

Appending data\disruptions-2011.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2012.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2013.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2014.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2015.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2016.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2017.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2018.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2019.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2020.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2021.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2022.csv to data/disruptions-2011-2023.csv
Appending data\disruptions-2023.csv to data/disruptions-2011-2023.csv
Appending data\services-2019.csv to data/services-2019-2025.csv
Appending data\services-20

### 2. Put stations data into `stations` table in DuckDB, treating it as constant file

In [61]:
con = duckdb.connect('data/rail.duckdb')
con.execute("""CREATE OR REPLACE TABLE stations AS 
            SELECT * 
            FROM 'data/stations-2023.csv'""")

print('stations rows count:', con.execute(
    'SELECT COUNT(*) FROM stations').fetchone()[0])

stations rows count: 591


### 3. Create tables `distances` and `distances_long`, following the tutorial

In [62]:
# Use DuckDB SQL function read_csv with nullstr
con.execute(
    """CREATE OR REPLACE TABLE distances AS
     FROM read_csv(
         'data/tariff-distances-2022.csv', nullstr='XXX'
         );"""
)

# Create the long form as in the tutorial
con.execute(
    """CREATE OR REPLACE TABLE distances_long AS
     UNPIVOT distances
     ON COLUMNS (* EXCLUDE station)
     INTO NAME other_station VALUE distance;"""
)

print('distances rows count:', con.execute(
    'SELECT COUNT(*) FROM distances').fetchone())
print('distances_long rows count:', con.execute(
    'SELECT COUNT(*) FROM distances_long').fetchone())

distances rows count: (399,)
distances_long rows count: (158802,)


### 4. Loading `disruptions` table, treating as OLTP

In [63]:
con.execute("""CREATE OR REPLACE TABLE disruptions AS 
            FROM read_csv_auto('data/disruptions-2011-2023.csv');""")
print('disruptions rows count:', con.execute(
    'SELECT COUNT(*) FROM disruptions').fetchone()[0])

disruptions rows count: 49900


### 5. Transform train services CSV files into a single Parquet file and make `services` table.

In [64]:
con.execute("""COPY (SELECT * FROM 'data/services-2019-2025.csv') 
            TO 'data/services-2019-2025.parquet' 
            (FORMAT parquet, COMPRESSION zstd);""")

# Register table in DuckDB from the written parquet
con.execute("""CREATE OR REPLACE TABLE services AS 
            SELECT * FROM read_parquet('data/services-2019-2025.parquet')""")

print('services rows count:', con.execute('SELECT COUNT(*) FROM services').fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

services rows count: 131795422


### 7.1. How many trains departed from Amsterdam Central station overall?

In [65]:
result = con.execute("""SELECT COUNT(*) FROM services 
    WHERE "Stop:Station name" = 'Amsterdam Centraal' 
    AND "Stop:Arrival time" IS NULL;
    """).fetchone()[0]
print('Number of trains departed from Amsterdam Central station:', result)

Number of trains departed from Amsterdam Central station: 975712


### 7.2. Calculate the average arrival delay of different service types (`Service:Type`). Order results descending by average delay.

In [66]:
result = con.execute("""
    SELECT "Service:Type", AVG("Stop:Arrival delay") as avg_delay
    FROM services
    GROUP BY "Service:Type"
    ORDER BY avg_delay DESC;
    """).fetchall()

for row in result:
    print(f"{row[0]}: {row[1]}")

Alpen Express: 31.84254143646409
Krokus Express: 18.304347826086957
European Sleeper: 12.788511187607574
Nightjet: 9.233219130191582
Eurostar: 6.926222070393817
Thalys: 5.3961896909446025
ICE International: 5.37118175477517
Int. Trein: 4.9513737892864205
Nachttrein: 4.155
Stoomtrein: 3.366863905325444
Dinner Train: 2.633956386292835
Intercity direct: 2.1796593823655344
Eurocity Direct: 2.1457246541165795
RE 19: 1.9698529411764707
Extra trein: 1.8759403965252088
EuroCity: 1.7368700265251988
Sneltrein: 1.1708414811098384
Train Charter: 1.16
Intercity: 0.8835712804698268
Speciale Trein: 0.6703727173224264
InnovationXpress: 0.6666666666666666
Stoptrein: 0.6082444607979275
Sprinter: 0.6060885928552995
Snelbus ipv trein: 0.012929163806232196
Stopbus ipv trein: 0.00029407178634248556
Stopbus i.p.v. trein: 1.1917994163946438e-05
Belbus ipv trein: 0.0
Belbus: 0.0
Tram i.p.v. trein: 0.0
Bus: 0.0
Metro ipv trein: 0.0
Metro: 0.0
Taxibus ipv trein: 0.0
stoptrein: 0.0
Metro i.p.v. trein: 0.0
Snelbus

### 7.3. What was the most common disruption cause in different years? [MODE function](https://duckdb.org/docs/stable/sql/functions/aggregates.html#modex) may be useful.

In [67]:
result = con.execute("""
       SELECT EXTRACT(YEAR FROM start_time) as year, 
              MODE(statistical_cause_en) as most_common_cause,
              COUNT(*) as occurrences
       FROM disruptions
       GROUP BY year
       ORDER BY year;
       """).fetchall()

for row in result:
    print(f"{int(row[0])}: {row[1]} ({row[2]} occurrences)")

2011: broken down train (1846 occurrences)
2012: points failure (2074 occurrences)
2013: points failure (2312 occurrences)
2014: broken down train (2484 occurrences)
2015: broken down train (2947 occurrences)
2016: broken down train (3031 occurrences)
2017: broken down train (4085 occurrences)
2018: broken down train (5190 occurrences)
2019: broken down train (5940 occurrences)
2020: broken down train (4450 occurrences)
2021: broken down train (4874 occurrences)
2022: broken down train (5499 occurrences)
2023: broken down train (5168 occurrences)


### 7.4. How many trains started their overall service in any Amsterdam station?

In [68]:
result = con.execute("""
    SELECT COUNT(DISTINCT "Service:RDT-ID") FROM services 
    WHERE "Stop:Station name" LIKE '%Amsterdam%' 
    AND "Stop:Arrival time" IS NULL;
    """).fetchone()[0]

print(f'Number of trains that started their service in any Amsterdam station {result}:')

Number of trains that started their service in any Amsterdam station 1100049:


### 7.5. What fraction of services was run to final destinations outside the Netherlands?

In [69]:
total_services = con.execute("""
    SELECT COUNT(DISTINCT "Service:RDT-ID") 
    FROM services;""").fetchone()[0]

outside_services = con.execute("""
    SELECT COUNT(DISTINCT s."Service:RDT-ID") 
    FROM services s 
    JOIN stations st ON s."Stop:Station code" = st.code 
    WHERE s."Stop:Departure time" IS NULL AND st.country != 'NL';
    """).fetchone()[0]

print(f'Fraction of services run to final destinations outside the Netherlands: {outside_services / total_services:.4f}')

Fraction of services run to final destinations outside the Netherlands: 0.0364


### 7.6. What is the largest distance between stations in the Netherlands (code `NL`)?

In [70]:
result = con.execute("""
    SELECT MAX(dl.distance) as max_distance
    FROM distances_long dl
    JOIN stations s1 ON dl.station = s1.code
    JOIN stations s2 ON dl.other_station = s2.code
    WHERE s1.country = 'NL' AND s2.country = 'NL' AND dl.distance IS NOT NULL;
    """).fetchone()[0]

print(f'Largest distance between stations in the Netherlands: {result} km')

Largest distance between stations in the Netherlands: 426 km


### 7.7. Compare the average arrival delay between different train operators (`Service:Company`) on a bar plot. Sort them appropriately.

In [71]:
result = con.execute("""
    SELECT "Service:Company" as company, AVG("Stop:Arrival delay") as avg_delay
    FROM services
    WHERE "Stop:Arrival time" IS NOT NULL
    GROUP BY "Service:Company"
    ORDER BY avg_delay DESC;
    """).fetchall()

df = pd.DataFrame(result, columns=['Company', 'Avg Delay'])
fig = px.bar(df, x='Company', y='Avg Delay', title='Average Arrival Delay by Train Operator')
fig.show()

### 7.8. How many services were disrupted in different years? Make a line plot.

In [72]:
result = con.execute("""
    SELECT EXTRACT(YEAR FROM start_time) as year, COUNT(DISTINCT rdt_id) as disrupted_services
    FROM disruptions
    GROUP BY year
    ORDER BY year;
    """).fetchall()

df = pd.DataFrame(result, columns=['Year', 'Disrupted Services'])
fig = px.line(df, x='Year', y='Disrupted Services', title='Number of Disrupted Services by Year')
fig.show()

### 7.9. What fraction of all services were cancelled (`Service:Completely cancelled`) in different years? Make a line plot.

In [73]:
# Get total services per year
total_df = pd.DataFrame(con.execute("""
    SELECT EXTRACT(YEAR FROM "Service:Date") as year, 
    COUNT(DISTINCT "Service:RDT-ID") as total_services
    FROM services
    GROUP BY year
    ORDER BY year;
    """).fetchall(), columns=['year', 'total_services'])

# Get cancelled services per year
cancelled_df = pd.DataFrame(con.execute("""
    SELECT EXTRACT(YEAR FROM "Service:Date") as year, 
    COUNT(DISTINCT "Service:RDT-ID") as cancelled_services
    FROM services
    WHERE "Service:Completely cancelled" = true
    GROUP BY year
    ORDER BY year;
    """).fetchall(), columns=['year', 'cancelled_services'])

# Merge and calculate fraction
df = pd.merge(total_df, cancelled_df, on='year', how='left').fillna(0)
df['fraction_cancelled'] = df['cancelled_services'] / df['total_services']

# Plot
fig = px.line(df, x='year', y='fraction_cancelled', title='Fraction of Services Cancelled by Year')
fig.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### 8.2. Create table `station_connections`, with columns `Service:RDT-ID`, `start_station_code` and `end_station_code` (pair of stations on a route), and `distance` between them.

In [74]:
# Using window functions to get next station in the route
con.execute("""
    CREATE OR REPLACE TEMP VIEW service_segments AS
    SELECT 
        "Service:RDT-ID",
        "Stop:Station code" AS start_station_code,
        LEAD("Stop:Station code") OVER (
            PARTITION BY "Service:RDT-ID" 
            ORDER BY "Stop:Departure time" NULLS LAST
        ) AS end_station_code
    FROM services
    WHERE "Stop:Station code" IS NOT NULL
    """)

In [75]:
# Create station_connections table with distances
# Key optimization: deduplicate BEFORE joining with distances table
# This drastically reduces the number of rows to process

con.execute("""
    CREATE OR REPLACE TABLE station_connections AS
    SELECT DISTINCT
        ss.start_station_code,
        ss.end_station_code,
        dl.distance
    FROM service_segments ss
    JOIN distances_long dl ON 
        ss.start_station_code = dl.station 
        AND ss.end_station_code = dl.other_station
    WHERE ss.end_station_code IS NOT NULL
        AND dl.distance IS NOT NULL
    """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### 8.3 What is the largest distance between a pair of stations?

In [76]:
# Find the largest distance between a pair of stations
result = con.execute("""
    SELECT start_station_code, end_station_code, distance
    FROM station_connections
    ORDER BY distance DESC
    LIMIT 1
    """).fetchone()

print(f'Largest distance between a pair of stations: {result[0]} -> {result[1]}: {result[2]} km')

Largest distance between a pair of stations: VS -> STV: 421 km


### 8.4. Histogram of inter-station distances

In [77]:
# Get distance data
df = pd.DataFrame(con.execute("""
    SELECT distance 
    FROM station_connections
    WHERE distance IS NOT NULL
    ORDER BY distance
    """).fetchall(), columns=['distance'])

# Create histogram
fig = px.histogram(
    df, 
    x='distance', 
    nbins=50,
    title='Histogram of Inter-Station Distances',
    labels={'distance': 'Distance (km)', 'count': 'Frequency'}
)
fig.update_layout(
    xaxis_title='Distance (km)',
    yaxis_title='Number of Station Pairs'
)
fig.show()